# 03: Bonus Extensions: KTA Optimization, Noise/Hardware Comparison, Blind Prediction

A thin interactive wrapper around `src.bonus_extensions`. `EXECUTION_CONFIG` controls where the quantum circuits below actually run.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

from src.config import CLASSIFICATION_TARGET, MODELING_TABLE_CSV
from src.features import get_candidate_matrix, get_modeling_features
from src.quantum_backend import ExecutionConfig, QuantumExecutor
from src.bonus_extensions import (
    KTAOptimizedQuantumKernel, execution_degradation_study, fit_deployment_model, BlindPredictor,
)
from src.classical_models import make_svc

EXECUTION_CONFIG = ExecutionConfig(mode="aer_simulator", shots=2048)

data = pd.read_csv(MODELING_TABLE_CSV)
X_full = get_candidate_matrix(data)
y = data[CLASSIFICATION_TARGET]
# Honors a MANUAL_FEATURES override in .env; falls back to automated selection.
selection = get_modeling_features(X_full, y)
X = data[selection["selected_features"]]
selection["selected_features"]

## Kernel Target Alignment optimization (trained quantum kernel)

In [ ]:
X_scaled = MinMaxScaler(feature_range=(0, np.pi)).fit_transform(X.values)
kta_executor = QuantumExecutor(EXECUTION_CONFIG)
kta_model = KTAOptimizedQuantumKernel(executor=kta_executor, n_layers=3, maxiter=60)
kta_model.fit(X_scaled, y.values)
print(f"KTA before: {kta_model.kta_before_:.4f} -> after: {kta_model.kta_after_:.4f}")

## Ideal vs noisy/real hardware degradation study

In [ ]:
# comparison_config defaults to a local device-like noise model. Swap in
# ExecutionConfig(mode="ibm_runtime") to compare against real hardware.
degradation_result = execution_degradation_study(X.values, y.values, feature_map_name="angle", n_splits=5)
degradation_result

## Blind SMILES prediction interface

In [ ]:
model, scaler = fit_deployment_model(make_svc, X, y.values)
blind = BlindPredictor(model, scaler, selection["selected_features"])

# Aspirin, just for illustration. Not one of the 29 training drugs.
# D50 and apparent_solubility are experimental measurements BlindPredictor
# cannot derive from SMILES alone, so supply placeholders for whichever of
# them the feature selection actually picked.
demo_experimental_values = {"D50": 50.0, "apparent_solubility": 10.0}
kwargs = {f: v for f, v in demo_experimental_values.items() if f in selection["selected_features"]}
blind.predict("CC(=O)OC1=CC=CC=C1C(=O)O", **kwargs)